In [ ]:
import requests
import lxml.html
from pypdf import PdfReader
import pandas as pd
from google import genai
from pathlib import Path
from google.genai import types
import os
import us
from datetime import datetime

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional

from ingest_sc_cases import case_df
from ingest_sc_judges import judge_pd

Scraped judge: Abigail LeGrow
Scraped judge: Adam Tanenbaum
Scraped judge: Aimee A. Oravec
Scraped judge: Alisa Kelli Wise
Scraped judge: Allison Riggs
Scraped judge: Andrew J. McDonald
Scraped judge: Andrew M. Horton
Scraped judge: Andrew M. Mead
Scraped judge: Andrew Pinson
Scraped judge: Angela M. Eaves
Scraped judge: Angela McCormick Bisig
Scraped judge: Anita Earls
Scraped judge: Ann A. Scott Timmer
Scraped judge: Anna Blackburne-Rigsby
Scraped judge: Anne K. McKeig
Scraped judge: Anne M. Patterson
Scraped judge: Annette Kingsland
  Ziegler
Scraped judge: Anthony Cannataro
Scraped judge: Aruna Masih
Scraped judge: Barbara A. Madsen
Scraped judge: Barbara Webb
Scraped judge: Benjamin A. Land
Scraped judge: Bert Richardson
Scraped judge: Beth Baker
Scraped judge: Brady E. Mendheim, Jr.
Scraped judge: Brett Busby
Scraped judge: Brian D. Boatright
Scraped judge: Brian Hagedorn
Scraped judge: Brian K. Zahra
Scraped judge: Briana Zamora
Scraped judge: Bronson James
Scraped judge: Bryan 

In [ ]:
# Cases with no opinion documents
case_df[~case_df["opinion_link"].str.contains("statecourt", na=False)]

,docket_no,title,state,date,type,pending,opinion_link
0,25-0139,Kalarchik v. State,Montana,"April 14, 2026",Civil Rights,False,https://statecourtreport.org/sites/default/fil...
15,OP 25-0858,Kendrick v. Knudsen,Montana,"February 27, 2026",Voting Rights and Elections,False,https://statecourtreport.org/sites/default/fil...
22,PR 23-0496,In the Matter of Austin Miles Knudsen,Montana,"December 31, 2025",Government Structure,False,https://statecourtreport.org/sites/default/fil...
32,DA 24-0039,Montanans Against Irresponsible Densification ...,Montana,"March 17, 2026",Civil Rights,False,https://statecourtreport.org/sites/default/fil...
39,DA 24-0250,State v. Alford,Montana,"August 5, 2025",Criminal Law,False,https://statecourtreport.org/sites/default/fil...
48,OP 25-0729,Montanans for Nonpartisan Courts v. Knudsen,Montana,"November 18, 2025",Voting Rights and Elections,False,https://statecourtreport.org/sites/default/fil...
50,DA 23-0462,City of Kalispell v. Doman,Montana,"February 10, 2026",Speech and Religion,False,https://statecourtreport.org/sites/default/fil...
82,DA 23-0229,State v. Herzog,Montana,"June 17, 2025",Criminal Law,False,https://statecourtreport.org/sites/default/fil...
99,DA 24-0501,Netzer v. State,Montana,"November 4, 2025",Civil Rights,False,https://statecourtreport.org/sites/default/fil...
160,DA 23-0648,Montana Environmental Information Center v. Of...,Montana,"May 29, 2025",Economic and Labor Rights,False,https://statecourtreport.org/sites/default/fil...


In [38]:
class Opinion(BaseModel):
    judge_name: str = Field(description="The full name of the judge giving an opinion.")
    opinion_summary: str = Field(
        description="An extremely brief description of the judge's opinion on the case."
    )


class Case(BaseModel):
    title: str = Field(description="The title of the case.")
    decision: str = Field(description="The final decision that was made for the case.")
    issue_debate: str = Field(description="The main issue being debated in the case.")
    judge_opinions: List[Opinion]

In [39]:
def read_opinion(pdf_link: str, model_id: str, client: genai.Client, prompt: str):
    resp = requests.get(pdf_link).content

    genai_resp = client.models.generate_content(
        model=model_id,
        contents=[types.Part.from_bytes(data=resp, mime_type="application/pdf"), prompt],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": Case.model_json_schema(),
        },
    )

    return Case.model_validate_json(genai_resp.text)

In [ ]:
def analyze_state_cases(
    case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_start: str, client_info: dict
):
    prompt = prompt_start + "$DATAFRAME$" + judge_df.to_markdown(index=False)
    num_cases = len(case_df)
    model_id = client_info["model_id"]
    client = client_info["client"]

    file_dic = {}
    for i in range(num_cases):
        print(f"Querying case {i}")
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        state = case_df.iloc[i]["state"]
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        pdf_link = case_df.iloc[i]["opinion_link"]

        case_ref = "_".join([docket_no, state, date])

        opinion_resp = read_opinion(pdf_link, model_id, client, prompt)

        file_dic[case_ref] = {}
        file_dic[case_ref]["pdf_link"] = pdf_link
        file_dic[case_ref]["response"] = opinion_resp.model_dump()

    return file_dic

In [ ]:
def apply_model(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    with open(prompt_path, "r") as prompt_file:
        prompt_start = prompt_file.read()

    load_dotenv(find_dotenv())

    gemini_key = os.getenv("GEMINI_API_KEY")
    client_info = {"model_id": "gemini-2.5-flash-lite", "client": genai.Client(api_key=gemini_key)}

    case_dic = analyze_state_cases(case_df, judge_df, prompt_start, client_info)

    return case_dic

In [ ]:
def state_opinion_table(case_dic: dict):
    opinion_table = {"case_id": [], "name": [], "opinion": []}

    for key in case_dic.keys():
        opinions = case_dic[key]["response"]
        num_opinions = len(opinions["judge_opinions"])

        for i in range(num_opinions):
            opinion_table["case_id"].append(key)
            judge_name = opinions["judge_opinions"][i]["judge_name"]
            opinion_table["name"].append(judge_name)
            opinion_summary = opinions["judge_opinions"][i]["opinion_summary"]
            opinion_table["opinion"].append(opinion_summary)

    return pd.DataFrame(opinion_table)

In [45]:
mon_cases = case_df[case_df["state"] == "Montana"].head(5)
mon_judges = judge_pd[judge_pd["state"] == "MT"]

mon_opinions = state_opinion_table(mon_cases, mon_judges, "prompt.txt")

Querying case 0
Querying case 1
Querying case 2
Querying case 3
Querying case 4


In [ ]:
def state_case_table(case_df: pd.DataFrame, case_dic: dict):
    case_table = {
        "case_id": [],
        "docket_no": [],
        "title": [],
        "state": [],
        "date": [],
        "type": [],
        "dispute_desc": [],
        "decision_outcome": [],
        "decision_status": [],
    }
    num_cases = len(case_df)

    for i in range(num_cases):
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        case_table["docket_no"].append(docket_no)
        state = case_df.iloc[i]["state"]
        case_table["state"].append(state)
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        case_table["date"].append(date)

        title = case_df.iloc[i]["title"]
        case_table["title"].append(title)
        type = case_df.iloc[i]["type"]
        case_table["type"].append(type)

        case_id = "_".join([docket_no, state, date])
        case_table["case_id"].append(case_id)

        response = case_dic[case_id]["response"]
        case_table["dispute_desc"].append(response["issue_debate"])
        case_table["decision_outcome"].append(response["decision"])

        status = case_df.iloc[i]["pending"]
        case_table["decision_status"].append(status)

    return pd.DataFrame(case_table)

In [ ]:
def produce_tables(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    states = case_df["state"].sort_values().unique()

    opinions = []
    cases = []

    for state in states:
        state_cases = case_df[case_df["state"] == state]

        state_abbr = us.states.lookup(state).abbr
        state_judges = judge_df[judge_df["state"] == state_abbr]

        case_dic = apply_model(state_cases, state_judges, prompt_path)

        opinions.append(state_opinion_table(case_dic))
        cases.append(state_case_table(state_cases, case_dic))

    rd = {"opinion_table": pd.concat(opinions), "case_table": pd.concat(cases)}

    return rd

,case_id,docket_no,title,state,date,type,dispute_desc,decision_outcome,decision_status
0,"2025-O-00879_Louisiana_October-24,-2025",2025-O-00879,In re Judge Jennifer Medley,Louisiana,"October-24,-2025",Judicial Selection and Administration,The main issue debated is whether Judge Medley...,Judge Medley was suspended for thirty days wit...,False
1,"2024-KK-00737_Louisiana_May-9,-2025",2024-KK-00737,State v. Davieontray Breax,Louisiana,"May-9,-2025",Criminal Law,The issue is whether prosecutors may join capi...,"The indictment was quashed, and the matter was...",False
2,"2024-CA-00995,-2024-CA-00996_Louisiana_March-2...","2024-CA-00995,-2024-CA-00996","Fremin v. Boyd Racing, LLC",Louisiana,"March-21,-2025",Civil Rights,"The issue debated is whether Act 437, which le...",The court affirmed the district court's judgme...,False
3,"2024-CC-00899_Louisiana_March-21,-2025",2024-CC-00899,Welch v. United Medical Healthwest-New Orleans,Louisiana,"March-21,-2025",Voting Rights and Elections,The main issue debated is whether La. R.S. 29:...,The trial court's denial of Welch's motion to ...,False
4,"2024-CD-00359_Louisiana_October-25,-2024",2024-CD-00359,Fisher v. Harter,Louisiana,"October-25,-2024",Government Structure,"The main issue is whether La. R.S. 13:4163, wh...",The Court reversed the district court's denial...,False
